# Hypothesis H3: Discount >30% and Profit Margin

## tl;dr

Run all cells after placing **`orders_cleaned.csv`** in the same folder as this notebook. The final cells automatically report whether H3 is supported, show the requested correlation matrix and scatter/regression plot, and generate a pricing recommendation.

> **Important:** Correlation tests the overall monotonic/linear relationship, but it does not directly test the threshold claim “above 30%.” Therefore, this notebook also performs a **one-sided two-group test** comparing orders above 30% with orders at or below 30%. This avoids treating correlation alone as proof of the stated hypothesis.

## Context & Methods

### Business question
Are orders discounted by more than 30% associated with lower profit margins?

### Definitions
- **Order discount:** sales-weighted average discount across an order's line items.
- **Order profit margin:** total order profit / total order sales.
- **High-discount order:** order discount **strictly greater than 30%**.
- **Significance level:** $\alpha=0.05$.

### Statistical design
1. Aggregate line-item data to one row per order (prevents large orders from receiving extra weight).
2. Describe profit margins for high- and lower-discount orders.
3. Select Pearson correlation when both variables are reasonably normal and lack extreme outliers; otherwise use Spearman correlation.
4. Directly test H3 with a one-sided Welch t-test when group distributions are reasonably normal; otherwise use a one-sided Mann–Whitney U test.
5. Report effect size and a 95% confidence interval for the mean-margin difference.

### Key assumptions and limitations
- Sales must be positive for a defined profit margin.
- A discount stored as 0–100 is converted to 0–1 automatically.
- Association is not causation: product mix, customer segment, region, or order size may confound the result.
- The regression line is descriptive and should not be extrapolated beyond observed discounts.

In [1]:
# 1. Setup
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path("../../data/cleaned/orders_clean.csv")
HIGH_DISCOUNT_THRESHOLD = 0.30
ALPHA = 0.05
RANDOM_SEED = 42


In [2]:
# 2. Load data and resolve common column-name variants
raw = pd.read_csv(DATA_PATH)

def normalized_name(name):
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())

normalized_columns = {normalized_name(c): c for c in raw.columns}

def resolve_column(candidates, required=True):
    for candidate in candidates:
        key = normalized_name(candidate)
        if key in normalized_columns:
            return normalized_columns[key]
    if required:
        raise KeyError(f"Expected one of {candidates}; available columns: {list(raw.columns)}")
    return None

order_col = resolve_column(["Order ID", "Order_ID", "OrderID", "order_number", "id"])
sales_col = resolve_column(["Sales", "Revenue", "Order Sales", "Amount"])
profit_col = resolve_column(["Profit", "Order Profit", "Net Profit"])
discount_col = resolve_column(["Discount", "Discount Percentage", "Discount Percent", "Discount_pct"])

print(f"Rows loaded: {len(raw):,}")
print({"order": order_col, "sales": sales_col, "profit": profit_col, "discount": discount_col})
display(raw[[order_col, sales_col, profit_col, discount_col]].head())

KeyError: "Expected one of ['Sales', 'Revenue', 'Order Sales', 'Amount']; available columns: ['order_id', 'order_date', 'customer_id', 'customer_name', 'customer_country', 'customer_segment', 'product_id', 'product_name', 'product_category', 'vendor_id', 'order_quantity', 'unit_price_usd', 'order_value_usd', 'discount_pct', 'cost_of_goods_usd', 'gross_margin_usd', 'profit_margin_pct', 'order_status', 'promised_delivery_date', 'actual_delivery_date', 'delivery_delay_days', 'fulfillment_channel', 'warehouse_id', 'shipment_id', 'return_reason', 'region', 'created_by', 'last_modified_date', 'future_order_date_flag', 'negative_order_value_flag', 'order_value_mismatch_flag', 'unexpected_missing_actual_delivery_flag', 'unexpected_return_reason_flag', 'unexpected_missing_warehouse_flag', 'unexpected_missing_shipment_flag', 'orphan_product_id_flag']"

In [ ]:
# 3. Clean inputs and aggregate to the order grain
work = raw[[order_col, sales_col, profit_col, discount_col]].copy()
work.columns = ["order_id", "sales", "profit", "discount"]

for column in ["sales", "profit", "discount"]:
    work[column] = pd.to_numeric(work[column], errors="coerce")